In [77]:
# pip install pydicom


In [1]:
# Multi_Label_Classification.ipynb

import os
import numpy as np
import pandas as pd
import pydicom
import matplotlib.pyplot as plt
import vision_models.constants as constants
from pydicom.pixel_data_handlers.util import apply_modality_lut
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from keras import layers, models, applications, losses

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import tensorflow as tf
from keras import layers, models, applications, losses
from sklearn.metrics import confusion_matrix, classification_report

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report


2024-08-13 19:06:05.927115: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
# Set paths
train_images_path = constants.TRAIN_DATA_PATH
train_df = pd.read_csv("NEW_EDA/train_dataset.csv")
val_df = pd.read_csv("NEW_EDA/val_dataset.csv")
test_df = pd.read_csv("NEW_EDA/test_dataset.csv")

# One-hot encode the 'condition' and 'level' columns separately
train_condition_dummies = pd.get_dummies(train_df['condition'], prefix='condition')
train_level_dummies = pd.get_dummies(train_df['level'], prefix='level')

val_condition_dummies = pd.get_dummies(val_df['condition'], prefix='condition')
val_level_dummies = pd.get_dummies(val_df['level'], prefix='level')

test_condition_dummies = pd.get_dummies(test_df['condition'], prefix='condition')
test_level_dummies = pd.get_dummies(test_df['level'], prefix='level')

# Concatenate the dummy columns back to the original DataFrame
train_df = pd.concat([train_df.drop(['condition', 'level'], axis=1), train_condition_dummies, train_level_dummies], axis=1)
val_df = pd.concat([val_df.drop(['condition', 'level'], axis=1), val_condition_dummies, val_level_dummies], axis=1)
test_df = pd.concat([test_df.drop(['condition', 'level'], axis=1), test_condition_dummies, test_level_dummies], axis=1)

# Now you have your DataFrame with one-hot encoded columns for 'condition' and 'level'


In [15]:
train_df = train_df.sample(frac=0.005, random_state=42)
val_df = val_df.sample(frac=0.005, random_state=42)
test_df = test_df.sample(frac=0.005, random_state=42)

train_df

,study_id,series_id,image_name,file_meta_version,sop_class_uid,sop_instance_uid,transfer_syntax_uid,implementation_class_uid,implementation_version_name,content_date,...,condition_Left Subarticular Stenosis,condition_Right Neural Foraminal Narrowing,condition_Right Subarticular Stenosis,condition_Spinal Canal Stenosis,level_L1/L2,level_L2/L3,level_L3/L4,level_L4/L5,level_L5/S1,image_path
72452,1734639980,3791568557,15.dcm,b'\x00\x01',1.2.840.10008.5.1.4.1.1.4.1,1734639980.1.15,1.2.840.10008.1.2.5,1.2.40.0.13.1.1.1,PYDICOM 2.4.2,20240503,...,False,False,False,False,False,False,False,False,True,/opt/dataset/train_images/1734639980/379156855...
115379,2744048264,2245335453,20.dcm,b'\x00\x01',1.2.840.10008.5.1.4.1.1.4.1,2744048264.1.20,1.2.840.10008.1.2.5,1.2.40.0.13.1.1.1,PYDICOM 2.4.2,20240503,...,False,False,False,False,False,False,False,False,False,/opt/dataset/train_images/2744048264/224533545...
21069,1525013622,1497771265,1.dcm,b'\x00\x01',1.2.840.10008.5.1.4.1.1.4.1,1525013622.1.1,1.2.840.10008.1.2.5,1.2.40.0.13.1.1.1,PYDICOM 2.4.2,20240503,...,False,False,False,False,False,False,False,False,False,/opt/dataset/train_images/1525013622/149777126...
121271,786544183,935806222,10.dcm,b'\x00\x01',1.2.840.10008.5.1.4.1.1.4.1,786544183.1.10,1.2.840.10008.1.2.5,1.2.40.0.13.1.1.1,PYDICOM 2.4.2,20240503,...,False,False,False,False,False,False,False,False,False,/opt/dataset/train_images/786544183/935806222/...
125474,3345734507,3663789245,4.dcm,b'\x00\x01',1.2.840.10008.5.1.4.1.1.4.1,3345734507.1.4,1.2.840.10008.1.2.5,1.2.40.0.13.1.1.1,PYDICOM 2.4.2,20240503,...,False,False,False,False,False,False,False,False,False,/opt/dataset/train_images/3345734507/366378924...
39153,647971523,464949558,12.dcm,b'\x00\x01',1.2.840.10008.5.1.4.1.1.4.1,647971523.1.12,1.2.840.10008.1.2.5,1.2.40.0.13.1.1.1,PYDICOM 2.4.2,20240503,...,False,False,False,False,False,False,False,False,True,/opt/dataset/train_images/647971523/464949558/...
122696,3931051195,3232004939,8.dcm,b'\x00\x01',1.2.840.10008.5.1.4.1.1.4.1,3931051195.1.8,1.2.840.10008.1.2.5,1.2.40.0.13.1.1.1,PYDICOM 2.4.2,20240503,...,False,False,False,True,False,False,False,True,False,/opt/dataset/train_images/3931051195/323200493...
86294,1879696087,2286103847,4.dcm,b'\x00\x01',1.2.840.10008.5.1.4.1.1.4.1,1879696087.1.4,1.2.840.10008.1.2.5,1.2.40.0.13.1.1.1,PYDICOM 2.4.2,20240503,...,False,False,False,False,False,False,False,False,False,/opt/dataset/train_images/1879696087/228610384...
13618,1440537582,3694546203,9.dcm,b'\x00\x01',1.2.840.10008.5.1.4.1.1.4.1,1440537582.1.9,1.2.840.10008.1.2.5,1.2.40.0.13.1.1.1,PYDICOM 2.4.2,20240503,...,False,False,False,False,False,False,False,False,False,/opt/dataset/train_images/1440537582/369454620...
127370,3872005714,816578444,13.dcm,b'\x00\x01',1.2.840.10008.5.1.4.1.1.4.1,3872005714.1.13,1.2.840.10008.1.2.5,1.2.40.0.13.1.1.1,PYDICOM 2.4.2,20240503,...,False,False,False,False,False,False,False,False,False,/opt/dataset/train_images/3872005714/816578444...


In [16]:


# Data Preparation
for df in [train_df, val_df, test_df]:
    df['image_path'] = df.apply(lambda row: os.path.join(train_images_path, str(row['study_id']), str(row['series_id']), f"{row['instance_number']}.dcm"), axis=1)


In [17]:
print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

(34, 47)
(4, 47)
(4, 47)


In [18]:

# Function to read and preprocess images using pydicom
def load_image(img_path, target_size=(224, 224)):
    # Load DICOM file
    dicom = pydicom.dcmread(img_path.numpy().decode('utf-8'))
    img = dicom.pixel_array
    
    # Normalize the image
    img = img / np.max(img)
    
    # Ensure img has 3 dimensions (Height, Width, Channels)
    if len(img.shape) == 2:  # Grayscale image
        img = np.expand_dims(img, axis=-1)  # Add channel dimension
    
    # Convert grayscale to RGB (3 channels) if needed
    if img.shape[-1] == 1:
        img = np.concatenate([img, img, img], axis=-1)
    
    # Convert to float32 immediately
    img = tf.convert_to_tensor(img, dtype=tf.float32)
    
    # Now resize the image
    img = tf.image.resize(img, target_size)
    
    # Normalize pixel values to [0, 1]
    img = img / 255.0
    
    return img

# Wrapper function to use with tf.data.Dataset.map
def load_image_wrapper(img_path, target_size=(224, 224)):
    img = tf.py_function(func=load_image, inp=[img_path], Tout=tf.float32)
    img.set_shape((target_size[0], target_size[1], 3))  # Explicitly set shape (224, 224, 3)
    return img

# Create a TensorFlow dataset
def create_tf_dataset(df, batch_size=32, is_training=True, predict_condition=True, predict_level=True):
    # Dynamically choose label columns based on what you want to predict
    if predict_condition and predict_level:
        label_columns = [col for col in df.columns if col.startswith('condition_') or col.startswith('level_')]
    elif predict_condition:
        label_columns = [col for col in df.columns if col.startswith('condition_')]
    elif predict_level:
        label_columns = [col for col in df.columns if col.startswith('level_')]
    else:
        raise ValueError("At least one of `predict_condition` or `predict_level` must be True.")
    
    dataset = tf.data.Dataset.from_tensor_slices((df['image_path'], df[label_columns]))
    dataset = dataset.map(lambda x, y: (load_image_wrapper(x), y), num_parallel_calls=tf.data.AUTOTUNE)
    
    # Debugging: Print shape of each element in the dataset
    def print_shapes(x, y):
        print(f"Image shape: {x.shape}, Label shape: {y.shape}")
        return x, y
    
    dataset = dataset.map(print_shapes)  # Add this line to print shapes
    
    if is_training:
        dataset = dataset.shuffle(buffer_size=1024)
    dataset = dataset.batch(batch_size).prefetch(buffer_size=tf.data.AUTOTUNE)
    return dataset

# Example of creating datasets
train_dataset_condition = create_tf_dataset(train_df, predict_condition=True, predict_level=False)
train_dataset_level = create_tf_dataset(train_df, predict_condition=False, predict_level=True)
train_dataset_both = create_tf_dataset(train_df, predict_condition=True, predict_level=True)



Image shape: (224, 224, 3), Label shape: (5,)
Image shape: (224, 224, 3), Label shape: (5,)
Image shape: (224, 224, 3), Label shape: (10,)


In [21]:


# Plot training history
def plot_metrics(history, title):
    plt.plot(history.history['accuracy'], label='Train Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.title(f'{title} Accuracy')
    plt.ylabel('Accuracy')
    plt.xlabel('Epoch')
    plt.legend()
    # plt.savefig(f'NEW_EDA/{title}.png')
    plt.show()

# Plot confusion matrix
def plot_confusion_matrix(y_true, y_pred, classes, title='Confusion Matrix', cmap=plt.cm.Blues):
    cm = confusion_matrix(y_true, y_pred)
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, xticklabels=classes, yticklabels=classes)
    
    plt.title(title)
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()

# Function to save predictions and actual labels to CSV
def save_predictions_to_csv(true_labels, predicted_labels, df, dataset_name, label_classes):
    # Map the indices back to their actual label names
    true_names = [label_classes[i] for i in true_labels]
    predicted_names = [label_classes[i] for i in predicted_labels]
    
    # Create a DataFrame with true and predicted labels
    results_df = pd.DataFrame({
        'image_path': df['image_path'],
        'true_label': true_names,
        'predicted_label': predicted_names
    })
    
    # # Save to CSV
    # results_df.to_csv(f'NEW_EDA/{dataset_name}_predictions.csv', index=False)
    # print(f"{dataset_name}_predictions.csv saved successfully!")

# Evaluate the model and plot the confusion matrix
def evaluate_model(model, dataset, df, dataset_name, condition_classes=None, level_classes=None, task_type='condition'):
    # Determine which label columns to use
    if task_type == 'condition':
        label_columns = [col for col in df.columns if col.startswith('condition_')]
        print("Condition columns used:", label_columns)
        predictions = model.predict(dataset)
        predicted_labels = np.argmax(predictions, axis=1)
        true_labels = np.argmax(df[label_columns].values, axis=1)
        evaluate_and_save(predicted_labels, true_labels, condition_classes, df, dataset_name)
    
    elif task_type == 'level':
        label_columns = [col for col in df.columns if col.startswith('level_')]
        print("Level columns used:", label_columns)
        predictions = model.predict(dataset)
        predicted_labels = np.argmax(predictions, axis=1)
        true_labels = np.argmax(df[label_columns].values, axis=1)
        evaluate_and_save(predicted_labels, true_labels, level_classes, df, dataset_name)
    
    elif task_type == 'condition_level':
        condition_columns = [col for col in df.columns if col.startswith('condition_')]
        level_columns = [col for col in df.columns if col.startswith('level_')]
        print("Condition columns used:", condition_columns)
        print("Level columns used:", level_columns)
        
        predictions = model.predict(dataset)
        condition_predicted_labels = np.argmax(predictions[0], axis=1)
        level_predicted_labels = np.argmax(predictions[1], axis=1)
        
        condition_true_labels = np.argmax(df[condition_columns].values, axis=1)
        level_true_labels = np.argmax(df[level_columns].values, axis=1)
        
        evaluate_and_save(condition_predicted_labels, condition_true_labels, condition_classes, df, dataset_name, 'condition')
        evaluate_and_save(level_predicted_labels, level_true_labels, level_classes, df, dataset_name, 'level')

def evaluate_and_save(predicted_labels, true_labels, label_classes, df, dataset_name, task_type='condition'):
    # Check the unique classes in both true and predicted labels
    unique_true_labels = set(true_labels)
    unique_pred_labels = set(predicted_labels)
    
    print(f"Unique true {task_type} labels: {unique_true_labels}")
    print(f"Unique predicted {task_type} labels: {unique_pred_labels}")
    
    print(f"Evaluation on {dataset_name} Data ({task_type.title()}):")
    try:
        print(classification_report(true_labels, predicted_labels, target_names=label_classes))
    except ValueError as e:
        print(f"Error during classification report generation: {e}")
        print(f"True labels: {true_labels}")
        print(f"Predicted labels: {predicted_labels}")
        print(f"Label classes: {label_classes}")
    
    # Plot confusion matrix
    plot_confusion_matrix(true_labels, predicted_labels, classes=label_classes, title=f'{dataset_name} {task_type.title()} Confusion Matrix')
    
    # Save predictions and actual labels to CSV
    save_predictions_to_csv(true_labels, predicted_labels, df, dataset_name + f'_{task_type}', label_classes)

# Function to create the CNN model
def create_cnn_model(task_type, num_condition_classes=None, num_level_classes=None):
    model = models.Sequential([
        layers.Input(shape=(224, 224, 3)),
        layers.Conv2D(32, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(128, activation='relu')
    ])
    
    if task_type == 'condition':
        model.add(layers.Dense(num_condition_classes, activation='softmax', name='condition_output'))
    elif task_type == 'level':
        model.add(layers.Dense(num_level_classes, activation='softmax', name='level_output'))
    elif task_type == 'condition_level':
        model.add(layers.Dense(num_condition_classes, activation='softmax', name='condition_output'))
        model.add(layers.Dense(num_level_classes, activation='softmax', name='level_output'))
    
    return model

# Function to create the ResNet model
def create_resnet_model(task_type, num_condition_classes=None, num_level_classes=None):
    base_model = applications.ResNet50(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
    base_model.trainable = False
    
    x = layers.GlobalAveragePooling2D()(base_model.output)
    x = layers.Dense(128, activation='relu')(x)
    
    condition_output = None
    level_output = None
    
    if task_type == 'condition':
        condition_output = layers.Dense(num_condition_classes, activation='softmax', name='condition_output')(x)
        model = models.Model(inputs=base_model.input, outputs=condition_output)
    elif task_type == 'level':
        level_output = layers.Dense(num_level_classes, activation='softmax', name='level_output')(x)
        model = models.Model(inputs=base_model.input, outputs=level_output)
    elif task_type == 'condition_level':
        condition_output = layers.Dense(num_condition_classes, activation='softmax', name='condition_output')(x)
        level_output = layers.Dense(num_level_classes, activation='softmax', name='level_output')(x)
        model = models.Model(inputs=base_model.input, outputs=[condition_output, level_output])
    
    return model

# Function to dynamically create and train a model
def run_model(task_type, model_type, train_df, val_df, test_df):
    num_condition_classes = len([col for col in train_df.columns if col.startswith('condition_')])
    num_level_classes = len([col for col in train_df.columns if col.startswith('level_')])

    if model_type == 'cnn':
        model = create_cnn_model(task_type, num_condition_classes, num_level_classes)
    elif model_type == 'resnet':
        model = create_resnet_model(task_type, num_condition_classes, num_level_classes)

    if task_type == 'condition':
        model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    elif task_type == 'level':
        model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    elif task_type == 'condition_level':
        model.compile(optimizer='adam', 
                      loss={'condition_output': 'categorical_crossentropy', 'level_output': 'categorical_crossentropy'},
                      metrics={'condition_output': 'accuracy', 'level_output': 'accuracy'})

    train_dataset = create_tf_dataset(train_df, predict_condition=(task_type in ['condition', 'condition_level']), predict_level=(task_type in ['level', 'condition_level']))
    val_dataset = create_tf_dataset(val_df, is_training=False, predict_condition=(task_type in ['condition', 'condition_level']), predict_level=(task_type in ['level', 'condition_level']))
    test_dataset = create_tf_dataset(test_df, is_training=False, predict_condition=(task_type in ['condition', 'condition_level']), predict_level=(task_type in ['level', 'condition_level']))

    history = model.fit(train_dataset, validation_data=val_dataset, epochs=2)
    
    title_suffix = f"{model_type.upper()} Model - {task_type.replace('_', ' & ').title()}"
    plot_metrics(history, f"Training History for {title_suffix}")

    # Evaluate and save predictions for validation and test datasets
    condition_classes = [col for col in val_df.columns if col.startswith('condition_')]
    level_classes = [col for col in val_df.columns if col.startswith('level_')]

    if task_type == 'condition_level':
        evaluate_model(model, val_dataset, val_df, "Validation", condition_classes, level_classes, task_type)
        evaluate_model(model, test_dataset, test_df, "Test", condition_classes, level_classes, task_type)
    else:
        label_classes = condition_classes if task_type == 'condition' else level_classes
        evaluate_model(model, val_dataset, val_df, "Validation", label_classes, task_type)
        evaluate_model(model, test_dataset, test_df, "Test", label_classes, task_type)


In [24]:


# Example usage:
run_model('condition', 'cnn', train_df, val_df, test_df)  # Predict condition using CNN
# run_model('level', 'resnet', train_df, val_df, test_df)    # Predict level using ResNet
# run_model('condition_level', 'cnn', train_df, val_df, test_df)  # Predict both condition and level using CNN


2024-08-13 20:18:33.116663: W external/local_tsl/tsl/framework/bfc_allocator.cc:487] Allocator (GPU_0_bfc) ran out of memory trying to allocate 3.4KiB (rounded to 3584)requested by op AddV2
If the cause is memory fragmentation maybe the environment variable 'TF_GPU_ALLOCATOR=cuda_malloc_async' will improve the situation. 
Current allocation summary follows.
Current allocation summary follows.
2024-08-13 20:18:33.116712: I external/local_tsl/tsl/framework/bfc_allocator.cc:1044] BFCAllocator dump for GPU_0_bfc
2024-08-13 20:18:33.116724: I external/local_tsl/tsl/framework/bfc_allocator.cc:1051] Bin (256): 	Total Chunks: 49, Chunks in use: 49. 12.2KiB allocated for chunks. 12.2KiB in use in bin. 2.8KiB client-requested in use in bin.
2024-08-13 20:18:33.116731: I external/local_tsl/tsl/framework/bfc_allocator.cc:1051] Bin (512): 	Total Chunks: 5, Chunks in use: 5. 2.5KiB allocated for chunks. 2.5KiB in use in bin. 2.3KiB client-requested in use in bin.
2024-08-13 20:18:33.116737: I extern

ResourceExhaustedError: {{function_node __wrapped__AddV2_device_/job:localhost/replica:0/task:0/device:GPU:0}} failed to allocate memory [Op:AddV2] name: 